In [4]:
#%uv pip install memory-profiler

In [1]:
import gc
gc.isenabled()

True

In [5]:
# Import necessary libraries
from pyspark.sql import SparkSession
#from memory_profiler import profile
import cProfile
from pyspark.sql.functions import col


#@profile
#def read_csv_with_spark():
    # Initialize Spark session
spark = SparkSession.builder \
    .appName("CSV Reader") \
    .master("local[*]") \
    .getOrCreate()

try:
    # Path to your CSV file - replace with your actual file path
    csv_file_path = "/Users/shaydabanihashemi/data/lake/bronze/utd19_u.csv"

    metrics = spark.read.option("header", "true") \
        .option("inferSchema", "true") \
        .csv(csv_file_path)
    metrics = metrics.filter(metrics.error.isNull())
    metrics = metrics.filter(~metrics.speed.isNull())

    csv_file_path = "/Users/shaydabanihashemi/data/lake/bronze/detectors_public.csv"

    detectors = spark.read.option("header", "true") \
        .option("inferSchema", "true") \
        .csv(csv_file_path)

    #Join data tables
    df = metrics.join(detectors, 'detid')

    df.write.mode("overwrite").parquet('/Users/shaydabanihashemi/data/lake/silver/traffic.parquet')

    # Display the inferred schema
    print("Schema of the DataFrame:")
    df.printSchema()

    # Show the first few rows of the data
    print("\nSample data:")
    df.show(30)

    # Perform some basic operations
    print("\nNumber of rows:", df.count())

    # Select specific columns
    print("\nSelecting specific columns:")
    df.select(df.columns[:2]).show(5)

    # Filter data
    print("\nFiltered data:")
    if "age" in df.columns:
        df.filter(col("age") > 25).show(5)

    # Group by and aggregate
    print("\nGrouped data:")
    if len(df.columns) > 1:
        df.groupBy(df.columns[0]).count().show(5)

finally:
    # Stop the Spark session
    #spark.stop()
    print("Spark session stopped - not")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/02 18:46:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Schema of the DataFrame:
root
 |-- detid: string (nullable = true)
 |-- day: date (nullable = true)
 |-- interval: integer (nullable = true)
 |-- flow: double (nullable = true)
 |-- occ: double (nullable = true)
 |-- error: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- speed: double (nullable = true)
 |-- length: double (nullable = true)
 |-- pos: double (nullable = true)
 |-- fclass: string (nullable = true)
 |-- road: string (nullable = true)
 |-- limit: string (nullable = true)
 |-- citycode: string (nullable = true)
 |-- lanes: integer (nullable = true)
 |-- linkid: integer (nullable = true)
 |-- long: double (nullable = true)
 |-- lat: double (nullable = true)


Sample data:


+-------+----------+--------+----+----+-----+----------+----------------+-----------------+-----------------+--------+---------------+-----+----------+-----+------+-----------------+----------------+
|  detid|       day|interval|flow| occ|error|      city|           speed|           length|              pos|  fclass|           road|limit|  citycode|lanes|linkid|             long|             lat|
+-------+----------+--------+----+----+-----+----------+----------------+-----------------+-----------------+--------+---------------+-----+----------+-----+------+-----------------+----------------+
|N11131D|2017-10-24|       0|12.0|NULL| NULL|birmingham|            80.0|0.216825515433334|0.055437967495632|tertiary|Bradford Street|   48|birmingham|    1|    33|-1.88938349650183|52.4747417635415|
|N11131D|2017-10-24|     300|12.0|NULL| NULL|birmingham|            80.0|0.216825515433334|0.055437967495632|tertiary|Bradford Street|   48|birmingham|    1|    33|-1.88938349650183|52.4747417635415|



Number of rows: 4579126

Selecting specific columns:


+-------+----------+
|  detid|       day|
+-------+----------+
|N11131D|2017-10-24|
|N11131D|2017-10-24|
|N11131D|2017-10-24|
|N11131D|2017-10-24|
|N11131D|2017-10-24|
+-------+----------+
only showing top 5 rows


Filtered data:

Grouped data:


+-------+-----+
|  detid|count|
+-------+-----+
|N17131Z|  813|
|N12151Y| 2146|
|N53312T| 1148|
|N33122S| 2470|
|N53151A| 1148|
+-------+-----+
only showing top 5 rows

Spark session stopped - not


In [6]:
#if __name__ == "__main__":
print("Starting Spark CSV reader")
#df = read_csv_with_spark()
#cProfile.run('read_csv_with_spark()')
print("Finished processing CSV file")

Starting Spark CSV reader
Finished processing CSV file


In [7]:
print("\nDistinct Values in Road and City Code")
df.select("road", "city").distinct().show()


Distinct Values in Road and City Code


+--------------------+----------+
|                road|      city|
+--------------------+----------+
|High Street Deritend|birmingham|
|  Ladywood Middleway|birmingham|
|         Duke Street|    bolton|
|         Park Street|birmingham|
|        Bristol Road|birmingham|
|St Chads Circus Q...|birmingham|
|             Digbeth|birmingham|
|         Bank Street|    bolton|
|     Sherlock Street|birmingham|
|     Pershore Street|birmingham|
|     St Georges Road|    bolton|
|          Kay Street|    bolton|
|          Folds Road|    bolton|
|       Old Snow Hill|birmingham|
|   St Georges Street|    bolton|
|Higher Bridge Street|    bolton|
|           Deansgate|    bolton|
|  Belgrave Middleway|birmingham|
|      Hockley Street|birmingham|
|           Queensway|birmingham|
+--------------------+----------+
only showing top 20 rows



In [9]:
long_lat = df.select("detid", "long", "lat")
long_lat.show()

+-------+-----------------+----------------+
|  detid|             long|             lat|
+-------+-----------------+----------------+
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-1.88938349650183|52.4747417635415|
|N11131D|-

In [10]:
df.write.mode("overwrite").parquet('/Users/shaydabanihashemi/data/lake/silver/long_lat.parquet')

In [11]:
df.select("citycode").distinct().show()

+----------+
|  citycode|
+----------+
|    bolton|
|birmingham|
| constance|
| groningen|
| innsbruck|
|manchester|
| rotterdam|
|     tokyo|
|strasbourg|
|     paris|
| melbourne|
| santander|
|    madrid|
|    torino|
|   vilnius|
|    taipeh|
|  toulouse|
+----------+



In [11]:
df.select("city").distinct().show()

+----------+
|      city|
+----------+
|    bolton|
|birmingham|
| constance|
| groningen|
| innsbruck|
|manchester|
| rotterdam|
|    torino|
+----------+



In [8]:
df.select("city", "citycode").distinct().show()

+----------+----------+
|      city|  citycode|
+----------+----------+
|    bolton|    bolton|
|birmingham|birmingham|
| constance| constance|
| groningen| groningen|
| innsbruck| innsbruck|
|manchester|manchester|
| rotterdam| rotterdam|
|    torino|  toulouse|
|    torino| santander|
|    torino|    torino|
|    torino|   vilnius|
|    torino|     tokyo|
|    torino|     paris|
|    torino| melbourne|
|    torino|    madrid|
|    torino|    taipeh|
|    torino|strasbourg|
+----------+----------+



citycodes = df.select("citycode").distinct().collect()
basepath = "/Users/shaydabanihashemi/data/lake/silver/"
citycode_values = [row['citycode'] for row in citycodes]
for city in citycode_values:
    file_path = (f"{basepath}{city}.parquet")
    df.filter(df["citycode"] == city).write.mode("overwrite").parquet(file_path)

In [12]:
from pyspark.sql.functions import col, desc, asc
df = df.orderBy(col("flow").desc()).show()

+-----+----------+--------+------+----------------+-----+------+-----+-----------------+-----------------+--------------+---------------+-----+--------+-----+------+-----------+----------+
|detid|       day|interval|  flow|             occ|error|  city|speed|           length|              pos|        fclass|           road|limit|citycode|lanes|linkid|       long|       lat|
+-----+----------+--------+------+----------------+-----+------+-----+-----------------+-----------------+--------------+---------------+-----+--------+-----+------+-----------+----------+
|  450|2016-10-06|   65700|4284.0|125.299795261772| NULL|torino|34.19|0.383225782194628|0.041359359416023|       primary| ?eimyni?ki? g.|   50| vilnius|    1|   328|  25.287459| 54.694576|
|  450|2016-10-06|   65700|4284.0|125.299795261772| NULL|torino|34.19|0.268104232970688|0.138596712672381|       primary|Corso Orbassano|    0|  torino|    3|    37|  7.6120407|45.0311341|
| 3209|2016-10-06|   29100|4140.0|72.6315789473684| NUL